# TCN-GRU-ISAB 公榜推理入口

规则原文约束：提交只能有一个 notebook；该 notebook 暴露评估入口；公榜加载已训练权重而不重训，私榜通过同目录训练脚本从零重训。上传器实测与队友成功包显示的 UTF-8 纯文本、50MB 和三文件平铺限制不是规则原文。

In [ ]:
from __future__ import annotations

import re
import warnings
from collections.abc import Mapping
from datetime import timedelta
from pathlib import Path

import dai
import numpy as np
import pandas as pd
import torch
from tcn_gru_isab_train import (
    BOOK_DEPTH,
    FEATURE_COLS,
    FEATURE_COUNT,
    PUBLIC_INSTRUMENTS_TABLE,
    build_dataset,
    load_model,
    prediction_calendar_buffer_days,
    public_bar_table,
)

MODEL_PATH = Path("tcn_gru_isab_model.json")
SCORE_COLUMNS = ["date", "instrument", "score"]
_TABLE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def _table_name(value, label):
    if not isinstance(value, str) or _TABLE.fullmatch(value) is None:
        raise ValueError(f"{label} must be a simple SQL table identifier")
    return value


def _date_bounds(start_date, end_date):
    start, end = pd.Timestamp(start_date).normalize(), pd.Timestamp(end_date).normalize()
    if pd.isna(start) or pd.isna(end) or start > end:
        raise ValueError(f"invalid evaluation date range: {start_date!r} to {end_date!r}")
    return start, end


def _injected_or_public(datasources, key, public_table):
    value = datasources.get(key)
    if value is None:
        warnings.warn(
            f"{key} not injected; available={list(datasources)}; falling back to public table",
            RuntimeWarning,
            stacklevel=2,
        )
        value = public_table
    return _table_name(value, key)


def _query_universe(table, start, end):
    end_exclusive = end + timedelta(days=1)
    result = dai.query(
        f"SELECT date, instrument FROM {table} ORDER BY date, instrument",
        filters={"date": [start.strftime("%Y-%m-%d"), end_exclusive.strftime("%Y-%m-%d")]},
    ).df()
    if set(result.columns) != {"date", "instrument"}:
        raise ValueError(f"instruments table columns invalid: {list(result.columns)}")
    universe = result.loc[:, ["date", "instrument"]].copy()
    universe["date"] = pd.to_datetime(universe["date"], errors="raise").dt.normalize()
    universe["instrument"] = universe["instrument"].astype(str)
    universe = universe.loc[universe["date"].between(start, end, inclusive="both")].reset_index(
        drop=True
    )
    if universe.empty:
        raise ValueError("injected instruments universe returned zero rows")
    if universe.duplicated(["date", "instrument"]).any():
        raise ValueError("injected instruments universe contains duplicate keys")
    return universe


def _validate_metadata(metadata, models):
    if list(metadata.get("feature_cols", [])) != FEATURE_COLS:
        raise ValueError("saved feature_cols differ from actual flattened fields")
    if int(metadata.get("feature_count", -1)) != FEATURE_COUNT:
        raise ValueError("saved feature_count differs from model input dimension")
    frequency = str(metadata.get("frequency", ""))
    public_bar_table(frequency)
    if int(metadata.get("book_depth", -1)) != BOOK_DEPTH:
        raise ValueError("saved book_depth contract mismatch")
    model_cfg = metadata.get("model_cfg", {})
    bar_end_times = list(metadata.get("bar_end_times", []))
    lookback = int(metadata.get("lookback_days", -1))
    if lookback < 1 or int(model_cfg.get("lookback_days", -2)) != lookback:
        raise ValueError("saved lookback_days/model_cfg contract mismatch")
    if int(model_cfg.get("n_bars", -1)) != len(bar_end_times):
        raise ValueError("saved n_bars/bar_end_times contract mismatch")
    contract = metadata.get("label_contract", {})
    primary = contract.get("primary_head")
    if not isinstance(primary, str) or any(model.primary_head != primary for model in models):
        raise ValueError("loaded models do not match label_contract.primary_head")
    if int(metadata.get("n_seeds", -1)) != len(models):
        raise ValueError("saved n_seeds differs from loaded models")


def _validate_dataset(dataset, requested_instruments, expected_bars):
    if list(dataset.get("feature_cols", [])) != FEATURE_COLS:
        raise ValueError("actual queried feature_cols differ from saved order")
    if int(dataset.get("feature_count", -1)) != FEATURE_COUNT:
        raise ValueError("actual feature_count differs from model input dimension")
    x = np.asarray(dataset["x"], dtype=np.float32)
    mask = np.asarray(dataset["mask"], dtype=bool)
    dates = np.asarray(dataset["trading_dates"], dtype="datetime64[D]")
    instruments = np.asarray(dataset["instruments"], dtype=object).astype(str)
    if (
        x.ndim != 4
        or x.shape[:2] != mask.shape
        or x.shape[2] != expected_bars
        or x.shape[-1] != FEATURE_COUNT
    ):
        raise ValueError(f"dataset tensor contract mismatch: x={x.shape}, mask={mask.shape}")
    if instruments.tolist() != list(requested_instruments):
        raise ValueError("build_dataset changed requested instrument order")
    return x, mask, dates, instruments


def _validate_output(output, universe, missing_by_day):
    if list(output.columns) != SCORE_COLUMNS:
        raise ValueError(f"output columns must be exactly {SCORE_COLUMNS}")
    if output.duplicated(["date", "instrument"]).any():
        raise ValueError("output contains duplicate keys")
    if not np.isfinite(output["score"].to_numpy(dtype=np.float64)).all():
        raise ValueError("output scores must all be finite")
    if (
        not output[["date", "instrument"]]
        .reset_index(drop=True)
        .equals(universe.reset_index(drop=True))
    ):
        raise ValueError("output keys are not row-aligned with the injected universe")
    if set(output["date"].unique()) != set(universe["date"].unique()):
        raise ValueError("output is missing one or more evaluation trading dates")
    if float(missing_by_day.max()) >= 0.40:
        raise ValueError(
            f"daily pre-fill missing ratio must be < 40%; max={missing_by_day.max():.4%}"
        )
    if not set(output["instrument"]).issubset(set(universe["instrument"])):
        raise ValueError("output contains instruments outside the injected universe")


def _cross_section_zscore(values):
    array = np.asarray(values, dtype=np.float32)
    finite = np.isfinite(array)
    output = np.zeros(array.shape, dtype=np.float32)
    if finite.sum() < 2:
        return output
    scale = float(np.std(array[finite], ddof=0))
    if not np.isfinite(scale) or scale <= np.finfo(np.float32).eps:
        return output
    output[finite] = (array[finite] - float(np.mean(array[finite]))) / scale
    return output


def main(datasources, start_date, end_date) -> pd.DataFrame:
    if not isinstance(datasources, Mapping):
        raise TypeError("datasources must be a mapping")
    print(f"datasources keys: {list(datasources.keys())}", flush=True)
    models, metadata = load_model(MODEL_PATH, device="cpu")
    _validate_metadata(metadata, models)
    frequency = str(metadata["frequency"])
    bar_key = f"bar{frequency}"
    bar_table = _injected_or_public(datasources, bar_key, public_bar_table(frequency))
    instruments_table = _injected_or_public(datasources, "instruments", PUBLIC_INSTRUMENTS_TABLE)
    start, end = _date_bounds(start_date, end_date)
    universe = _query_universe(instruments_table, start, end)
    instrument_order = pd.unique(universe["instrument"]).tolist()
    stats = {"mean": metadata["mean"], "std": metadata["std"]}
    lookback = int(metadata["lookback_days"])
    bar_end_times = list(metadata["bar_end_times"])
    calendar_buffer = prediction_calendar_buffer_days(lookback)
    history_start = start - timedelta(days=calendar_buffer)
    print(
        f"history query: requested_lookback={lookback} trading days, "
        f"calendar_buffer={calendar_buffer} days, range={history_start.date()}..{end.date()}",
        flush=True,
    )
    dataset = build_dataset(
        bar_table,
        history_start,
        end,
        "predict",
        instrument_order,
        stats,
        trading_dates=universe["date"].unique(),
        frequency=frequency,
        bar_end_times=bar_end_times,
    )
    x, mask, trading_dates, instruments = _validate_dataset(
        dataset, instrument_order, len(bar_end_times)
    )
    first_prediction_day = np.datetime64(start.date(), "D")
    actual_history_days = int(np.unique(trading_dates[trading_dates < first_prediction_day]).size)
    print(
        f"history result: requested_lookback={lookback} trading days, "
        f"calendar_buffer={calendar_buffer} days, "
        f"actual_history_trading_days={actual_history_days}",
        flush=True,
    )
    if actual_history_days < lookback - 1:
        raise ValueError(
            f"insufficient prediction history: requested lookback={lookback}, "
            f"requires at least {lookback - 1} prior trading days, "
            f"obtained {actual_history_days} from a {calendar_buffer}-day calendar buffer"
        )
    date_to_index = {value: index for index, value in enumerate(trading_dates)}
    stock_to_index = {value: index for index, value in enumerate(instruments)}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = [model.to(device).eval() for model in models]
    score_rows = []
    with torch.inference_mode():
        for day, day_universe in universe.groupby("date", sort=False):
            date_key = np.datetime64(pd.Timestamp(day).date(), "D")
            if date_key not in date_to_index:
                continue
            date_index = date_to_index[date_key]
            if date_index < lookback - 1:
                continue
            stock_indices = np.asarray(
                [stock_to_index[value] for value in day_universe["instrument"]], dtype=np.int64
            )
            valid = mask[date_index, stock_indices]
            if not valid.any():
                continue
            selected = stock_indices[valid]
            window = x[date_index - lookback + 1 : date_index + 1, selected]
            tensor = torch.from_numpy(np.ascontiguousarray(window.transpose(1, 0, 2, 3))).to(device)
            per_seed = [
                model(tensor).detach().float().cpu().numpy().reshape(-1) for model in models
            ]
            if len(per_seed) == 1:
                combined = per_seed[0]
            else:
                normalized = [_cross_section_zscore(values) for values in per_seed]
                combined = np.mean(np.stack(normalized), axis=0, dtype=np.float32)
            valid_instruments = day_universe.loc[valid, "instrument"].tolist()
            score_rows.extend(
                {"date": day, "instrument": name, "score": float(score)}
                for name, score in zip(valid_instruments, combined, strict=True)
            )
    raw = pd.DataFrame(score_rows, columns=SCORE_COLUMNS)
    output = universe.merge(
        raw, on=["date", "instrument"], how="left", validate="one_to_one", sort=False
    )
    missing_by_day = (
        output["score"].replace([np.inf, -np.inf], np.nan).isna().groupby(output["date"]).mean()
    )
    output["score"] = (
        output["score"].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)
    )
    output = output.loc[:, SCORE_COLUMNS]
    _validate_output(output, universe, missing_by_day)
    print(
        f"prediction complete: frequency={frequency}, rows={len(output)}, "
        f"dates={output['date'].nunique()}, seeds={len(models)}, device={device}",
        flush=True,
    )
    return output